In [1]:
!pip install numpy>=2.0.0
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!pip install transformers==4.41.0
!pip install biopython==1.83

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
!python3 prepare_data.py \
    --input_files /content/AntiMRSA_only_DosTercios.fasta \
    --output_file_train=train_data_3_fams.txt \
    --output_file_test=test_data_3_fams.txt \
    --train_split_ratio=0.8 \
    --bidirectional

INFO: Loaded 482 sequences from /content/AntiMRSA_only_DosTercios.fasta
INFO: Data is bidirectional. Each sequence will be stored in both directions.
INFO: Train data: 385 sequences
INFO: Test data: 97 sequences
INFO: Saving training data to train_data_3_fams.txt
INFO: Saving test data to test_data_3_fams.txt


In [3]:
!python3 finetune.py \
    --model=hugohrban/progen2-small \
    --train_file=train_data_3_fams.txt \
    --test_file=test_data_3_fams.txt \
    --device=cuda \
    --epochs=15 \
    --batch_size=8 \
    --accumulation_steps=16 \
    --lr=5e-5 \
    --decay=plateau \
    --warmup_steps=100 \
    --eval_before_train

tokenizer.json: 1.63kB [00:00, 4.06MB/s]
INFO: Device: cuda
config.json: 1.16kB [00:00, 6.20MB/s]
configuration_progen.py: 2.63kB [00:00, 11.0MB/s]
A new version of the following files was downloaded from https://huggingface.co/hugohrban/progen2-small:
- configuration_progen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
modeling_progen.py: 24.6kB [00:00, 66.7MB/s]
A new version of the following files was downloaded from https://huggingface.co/hugohrban/progen2-small:
- modeling_progen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
model.safetensors: 100% 605M/605M [00:07<00:00, 81.4MB/s]
generation_config.json: 100% 111/111 [00:00<00:00, 935kB/s]
INFO: No new embeddings to initialize.
INFO: Runnning evaluation on test set before training...
100% 97/97 [00:02<00:00, 37.06it/s]


In [4]:
!python3 sample.py \
  --model=hugohrban/progen2-small \
  --device=cuda \
  --batch_size=10000 \
  --prompt="1" \
  --iters=1 \
  --min_length=15 \
  --max_length=34 \
  --k=20 \
  --t=1.2

INFO: Device: cuda
INFO: Loading model from hugohrban/progen2-small
INFO: Loading tokenizer
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
INFO: Prompt tokens: ['1']
INFO: Sampling batch 1 / 1
100% 16/16 [00:02<00:00,  7.79it/s]
2025-09-02 09:10:01.547215: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-02 09:10:01.565661: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756804201.587668  